In [1]:
import numpy as np

from model import TronBatchModel
from controller import RandomController, GreedySpaceController

import trainer

In [2]:
# Environment parameters
WIDTH = 64
HEIGHT = 48


## 1. Smoke test the vectorized environment

In [3]:
env = TronBatchModel(width=WIDTH, height=HEIGHT, players=2, envs=1024, keep_owner=False, seed=0)
ctrl = RandomController(seed=0)

obs = env.observe_lite()
actions = ctrl.actions(env)
result = env.step(actions)

print("obs shape:", obs.shape)
print("actions shape:", actions.shape)
print("reward shape:", result.reward.shape)
print("done count:", result.done.sum())

obs shape: (1024, 2, 17)
actions shape: (1024, 2)
reward shape: (1024, 2)
done count: 0


## 2. Fast rollout helper

trainer.evaluate_controller evaluates a controller over many parallel games.
Use this for baselines, genetic algorithms, and quick policy comparisons.

In [4]:

print("Random:", trainer.evaluate_controller(RandomController(1), envs=2048))
print("Greedy:", trainer.evaluate_controller(GreedySpaceController(), envs=2048))

Random: {'mean_reward_per_player': array([-0.04931641, -0.41748047], dtype=float32), 'win_rate_per_player': array([0.9995117, 0.9995117], dtype=float32), 'mean_length': np.float64(499.50439453125)}
Greedy: {'mean_reward_per_player': array([-0.60058594, -0.6308594 ], dtype=float32), 'win_rate_per_player': array([0.51904297, 0.51708984], dtype=float32), 'mean_length': np.float64(510.1474609375)}


## Params

Prefer currently
hidden: 8
popsize and elite_count: 64-8

In [5]:
greedy_opponent = GreedySpaceController()

genetic_algo_params = {
    "hidden": 16,
    'generations': 12,
    'pop_size': 32,
    'elite_count': 6,
    'eval_envs': 1024,

    "players":2,
    "width": WIDTH,
    "height": HEIGHT,
    'seed': 42,

}
ga_params = {
    **genetic_algo_params,
}

In [6]:
probe_env = TronBatchModel(
    width=genetic_algo_params["width"],
    height=genetic_algo_params["height"],
    players=genetic_algo_params["players"],
    envs=1)
obs_dim = probe_env.observe_lite().shape[-1]

policy = trainer.MLPPolicy(obs_dim, hidden=genetic_algo_params["hidden"], rng=0)
print("obs_dim:", obs_dim)
print("genome parameters:", policy.n_params)

obs_dim: 17
genome parameters: 339


## 4. Genetic algorithm starter

This is intentionally simple:

1. Keep a population of MLP genomes.
2. Evaluate each genome.
3. Keep elites.
4. Refill the population with mutation + crossover.

In [ ]:
best_fitness, best_genome, obs_dim = trainer.run_ga(
    **genetic_algo_params
)

print("best fitness:", best_fitness)

## 5. Save and reload a trained genome

This saves the best genome into `./tron_genomes/` using a filename that includes key parameters and the current date/time. It also writes a `.json` metadata sidecar with the training settings.


In [ ]:
import numpy as np

temp_params = {
    "hidden": ga_params["hidden"],
    'generations': ga_params["generations"],
    'pop_size': ga_params["pop_size"],
    'elite_count': ga_params["elite_count"],
    'eval_envs': ga_params["eval_envs"],

    "players": ga_params["players"],
    "width": ga_params["width"],
    "height": ga_params["height"],
    'seed': 42,
}
genome_path, metadata_path = trainer.export_genome(
    best_genome,
    obs_dim=obs_dim,
    fitness=best_fitness,
    **temp_params,
)

loaded = np.load(genome_path)
trained_policy = trainer.MLPPolicy(obs_dim, hidden=ga_params["hidden"], genome=loaded)

print(trainer.evaluate_controller(trained_policy, envs=ga_params["eval_envs"]))


## 6. Watch the trained genome in the GUI

The current `play_live()` function creates its own controller, so for visualizing a custom trained policy,
use a tiny local loop like this. Run it only on a machine with tkinter GUI support.

In [ ]:
from view import GameView
def watch_mixed_policy(policies, players=2, width=32, height=32, scale=16, fps=20, seed=0):
    board_model = TronBatchModel(width=width, height=height, players=players, envs=1, keep_owner=True, randomize_spawns=True, seed=seed)
    view = GameView(board_model, scale=scale, fps=fps)

    try:
        while view.poll():
            if view.take_restart_request():
                board_model.reset()
            if not board_model.done[0]:
                acts = np.zeros((1, board_model.players), dtype=np.int8)
                for i in range(len(policies)):
                    policy = policies[i]

                    policy_acts = policy.actions(board_model)
                    acts[:, i] = policy_acts[:, i]
                board_model.step(acts)
            view.render(board_model)
    finally:
        view.close()

In [ ]:
from src.trainer import watch_policy

watch_policy(trained_policy, width=ga_params["width"], height=ga_params["height"])